# Exercise 4 — The daily revenue pipeline

⏱️ 60 minutes hands-on. Work top to bottom and don't skip ahead to a later day.

## The situation

From Sofia, Head of Ops, after the quarterly review:

> Every morning we get an export of yesterday's orders, and every morning someone opens a
> spreadsheet and works out revenue by region by hand. I want that on a dashboard instead,
> refreshed daily, without anyone touching it.
>
> It has to be right. If the number on that dashboard is wrong, people will make decisions on
> it before anyone notices.

Three days of exports are waiting in `raw/batches/`. Process them one at a time, in order, as
if each one landed the morning it arrived.

In [ ]:
from pyspark.sql.functions import col
from northtrail import get_spark, path, local_data, remove, where_am_i

print(where_am_i())
spark = get_spark("exercise-4")

BRONZE = path("bronze", "orders")
SILVER = path("silver", "orders")
GOLD   = path("gold", "revenue_by_region")

def batch(day, filename="orders.parquet"):
    """Yesterday's export, as it landed."""
    return spark.read.parquet(local_data("lake", "raw", "batches", day, filename))

# Start from an empty slate. Re-running this notebook later starts day 1 over, which
# matters here -- you'll be reading this pipeline's own history before the day is out.
remove(spark, BRONZE, SILVER, GOLD)
print("ready")

---
## Day 1

The first export lands. Have a look at what's actually in it before you build anything.

In [ ]:
day1 = batch("day1")
print(f"{day1.count()} rows")
day1.show(5)

# Anything in here you wouldn't want on Sofia's dashboard?
day1.groupBy("order_id").count().filter("count > 1").show(5)
day1.filter("customer_id = 'TEST'").show()

400 real orders, 8 of them exported twice, and 2 QA rows from a release test.

Build it in three steps. Keep the raw export exactly as it arrived in one place, keep a
cleaned version in another, and keep Sofia's actual numbers in a third.

In [ ]:
# Step 1 -- land the export untouched. Nothing is filtered, nothing is fixed.
day1.write.format("delta").mode("overwrite").save(BRONZE)

print(f"bronze: {spark.read.format('delta').load(BRONZE).count()} rows")

In [ ]:
# Steps 2 and 3 -- this is the daily pipeline. You'll re-run it every day from here on.
def rebuild_silver_and_gold():
    # Silver: one row per order (most recent wins), no test data, amount typed as a number.
    spark.sql(f'''
        CREATE OR REPLACE TEMP VIEW ranked AS
        SELECT *, row_number() OVER (PARTITION BY order_id ORDER BY order_ts DESC) AS rn
        FROM delta.`{BRONZE}`
    ''')
    (spark.sql("SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1 AND customer_id <> 'TEST'")
          .withColumn("amount", col("amount").cast("double"))
          .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER))

    # Gold: what Sofia's dashboard reads.
    (spark.sql(f"SELECT region, round(sum(amount), 2) AS revenue, count(*) AS orders "
               f"FROM delta.`{SILVER}` GROUP BY region")
          .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD))

def gold():
    return spark.read.format("delta").load(GOLD).orderBy("region")

rebuild_silver_and_gold()
gold().show()

In [ ]:
spark.sql(f"SELECT round(sum(revenue), 2) AS total_revenue, sum(orders) AS orders FROM delta.`{GOLD}`").show()

## 💡 Concept: the three layers

You just built the standard shape for this, usually called **Bronze / Silver / Gold** (or a
"medallion" layout).

- **Bronze** is the export exactly as it arrived, appended to and never edited.
- **Silver** is the cleaned version: deduplicated, typed, test rows gone.
- **Gold** is the answer to one business question, in this case revenue by region.

The reason it's three tables instead of one script that goes straight from file to dashboard:
each layer can be rebuilt from the one before it. Silver and Gold are disposable — delete
them and you can regenerate both. Bronze is the only thing that can't be recreated, which is
why nothing is ever cleaned on the way in.

Whether that actually helps is not obvious yet. Keep it in mind.

---
## Day 2

Next morning, next export. Same job: land it in Bronze, re-run the pipeline.

In [ ]:
try:
    batch("day2").write.format("delta").mode("append").save(BRONZE)
    print("appended")
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:700]}")

## 🔍 What just happened?

The write refused. Read the error: it's comparing the schema of what you're writing against
the schema of the table, and they don't match — the export has grown a column.

In [ ]:
print("bronze :", spark.read.format("delta").load(BRONZE).columns)
print("day 2  :", batch("day2").columns)

batch("day2").select("order_id", "amount", "discount_code").show(5)

## 💡 Concept: schema evolution

Marketing launched promo codes overnight, so the export has a `discount_code` column that
didn't exist yesterday. Handling that is **schema evolution**, and Delta makes you ask for it
explicitly with `mergeSchema`, which adds the new column and backfills null for every existing
row.

Note what it did by default: **it stopped.** That is the useful behaviour. The alternative —
quietly dropping the column it didn't recognise — would have worked this morning and lost data
every morning after, and nobody would have found out until someone asked about discounts.

Allow it when a new column is genuinely additive, as here. Don't allow it blindly on a
pipeline where an unexpected column means something upstream is broken.

In [ ]:
batch("day2").write.format("delta").mode("append").option("mergeSchema", "true").save(BRONZE)

rebuild_silver_and_gold()
gold().show()
spark.sql(f"SELECT round(sum(revenue), 2) AS total_revenue, sum(orders) AS orders FROM delta.`{GOLD}`").show()

---
## 🚨 Incident

Day 3. You land the export and re-run the pipeline exactly as you did yesterday.

In [ ]:
batch("day3").write.format("delta").mode("append").option("mergeSchema", "true").save(BRONZE)

rebuild_silver_and_gold()
gold().show()
spark.sql(f"SELECT round(sum(revenue), 2) AS total_revenue, sum(orders) AS orders FROM delta.`{GOLD}`").show()

Sofia, 09:14:

> Did we do four million euros yesterday?

Yesterday's total was €99,208.71. Find out what landed.

In [ ]:
# Compare the day 3 export against a day you trust.
for day in ["day2", "day3"]:
    b = batch(day)
    b.selectExpr(f"'{day}' AS day", "count(*) AS rows", "count(distinct order_id) AS distinct_orders",
                 "round(min(amount), 2) AS min_amount", "round(max(amount), 2) AS max_amount").show()

## 🔍 What just happened?

Two things wrong with one export. 900 rows for 300 orders — every order is in there three
times — and the amounts are about a hundred times too big. Yesterday's largest order was
€795; this file has orders in the tens of thousands.

Upstream confirms it: a bad deploy sent amounts in cents instead of euros, and its retry loop
shipped the batch three times. They've re-exported the batch correctly — it's sitting next to
the bad one as `orders_fixed.parquet`.

So: load the corrected file and re-run the pipeline.

In [ ]:
batch("day3", "orders_fixed.parquet").write.format("delta").mode("append").option("mergeSchema", "true").save(BRONZE)

rebuild_silver_and_gold()
spark.sql(f"SELECT round(sum(revenue), 2) AS total_revenue, sum(orders) AS orders FROM delta.`{GOLD}`").show()

## 🔍 What just happened?

**Nothing changed.** Still €4,108,801.71.

Both versions of every day-3 order are now sitting in Bronze — the corrupted rows and the
corrected ones, same `order_id`, same `order_ts`. Your Silver step picks one row per order
and it has no way to tell which is which, so it keeps picking the wrong ones.

And you've made it worse: Bronze now holds four copies of every day-3 order. Appending is the
only way you've ever written to this table, and appending cannot remove anything.

Sofia is still waiting. The dashboard is still wrong.

## 💡 Concept: time travel

In Exercise 3 you saw that this table is a numbered sequence of commits, and each commit lists
which files are part of the table. Nothing was overwritten when you appended — the older
versions of the file list are all still there.

So the table as it stood *before* the bad batch is not gone and does not need restoring from a
backup. It is a version you can read directly (`VERSION AS OF`), or make current again
(`RESTORE`). This is **time travel**, and it works because the log never throws anything away.

Find the last version where Bronze was still correct.

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE}`").select("version", "operation", "operationMetrics").show(truncate=60)

In [ ]:
# Check a version before committing to it: how many rows, and what was the largest order?
for v in [1, 2]:
    df = spark.read.format("delta").option("versionAsOf", v).load(BRONZE)
    print(f"version {v}: {df.count():>5} rows, max amount {df.selectExpr('max(amount)').collect()[0][0]}")

In [ ]:
# Version 1 is Bronze with day 1 and day 2 in it, and no day 3 at all.
spark.sql(f"RESTORE TABLE delta.`{BRONZE}` TO VERSION AS OF 1")

print(f"bronze: {spark.read.format('delta').load(BRONZE).count()} rows")
rebuild_silver_and_gold()
spark.sql(f"SELECT round(sum(revenue), 2) AS total_revenue, sum(orders) AS orders FROM delta.`{GOLD}`").show()

Back to €99,208.71 — Bronze is exactly as it was before day 3 landed, and Gold agrees.

Day 3 still has to go in, though, and if you append the corrected file you're one accidental
re-run away from the same mess. Whatever you use has to be safe to run twice.

## 💡 Concept: idempotent reprocessing with MERGE

An append is not safe to repeat: run it twice, get the rows twice. `MERGE INTO` is — it
matches incoming rows against existing ones on a key you choose, updates the ones that already
exist and inserts only the ones that don't.

That property has a name worth knowing: the operation is **idempotent**, meaning running it
once and running it five times leave the table in the same state. It matters far more than it
sounds, because in production you do not control how many times a job runs. Retries, restarts,
a colleague re-running yesterday's backfill — all of them are normal, and all of them are
harmless against a merge.

In [ ]:
batch("day3", "orders_fixed.parquet").createOrReplaceTempView("incoming")

spark.sql(f'''
    MERGE INTO delta.`{BRONZE}` AS target
    USING incoming AS source
      ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
''')

rebuild_silver_and_gold()
gold().show()
spark.sql(f"SELECT round(sum(revenue), 2) AS total_revenue, sum(orders) AS orders FROM delta.`{GOLD}`").show()

In [ ]:
# The thing you could not do with an append: run it again.
spark.sql(f'''
    MERGE INTO delta.`{BRONZE}` AS target
    USING incoming AS source
      ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
''')

rebuild_silver_and_gold()
spark.sql(f"SELECT round(sum(revenue), 2) AS total_revenue, sum(orders) AS orders FROM delta.`{GOLD}`").show()

## Debrief

- You recovered by rolling Bronze back and replaying. Walk through what the same morning would
  have looked like if Bronze didn't exist and day 3 had been loaded straight into a cleaned
  table — what would you have had to go back to?
- Time travel here is a side effect of how the table records changes, not a backup someone
  configured. What does that get you that a nightly backup doesn't, and where would you still
  want the backup?
- Sofia's real complaint is that a wrong number reached the dashboard at all. Nothing you
  built today would have stopped that. Where in these three layers would you put a check, and
  what would you have it do when it fires?
- `MERGE` is safe to re-run and `append` isn't, so why not just use `MERGE` everywhere?